# optimizer-class-dispatch — faded example 3: Sweep multiple optimizers via dispatch and collect results

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-class-dispatch`. The last cell reports your progress on the `Config: Optimizer class dispatch` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: Optimizer class dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-class-dispatch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-class-dispatch"
DD_SUBTOPIC = "Config: Optimizer class dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When comparing optimizer configurations in a sweep, iterating over a dispatch dictionary gives you each `(name, class)` pair cleanly. Building each optimizer from the factory and running a training step makes the sweep code compact and uniform across all optimizer types.

## Faded exercise 3

The `OPTIM_MAP` and `build_optimizer` are defined. Complete `run_sweep(param_tensor, lr, steps)`: for each name in `OPTIM_MAP`, build an optimizer around `[param_tensor]`, run `steps` gradient steps (using a fixed gradient of `t.ones_like(param_tensor)`), and return a dict mapping optimizer name to the final parameter value as a tensor.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

OPTIM_MAP = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def build_optimizer(name: str, params, lr: float) -> t.optim.Optimizer:
    return OPTIM_MAP[name](params, lr=lr)

def run_sweep(param_init: t.Tensor, lr: float, steps: int) -> dict:
    results = {}
    for name in OPTIM_MAP:
        p = param_init.clone().detach().requires_grad_(True)
        opt = None  # TODO: fill in this step — read the prompt cell above
        for _ in range(steps):
            opt.zero_grad()
            p.grad = t.ones_like(p)
            opt.step()
        results[name] = p.detach().clone()
    return results

# --- run it ---
t.manual_seed(0)
init = t.tensor([1.0, 1.0])
results = run_sweep(init, lr=0.1, steps=3)
for name, val in results.items():
    print(f'{name}: {val}')


def _test():
    import torch as t
    init = t.tensor([1.0, 1.0])
    results = run_sweep(init, lr=0.1, steps=3)
    assert set(results.keys()) == set(OPTIM_MAP.keys()), 'Results must have one entry per optimizer'
    for name, val in results.items():
        assert val.shape == (2,), f'{name}: wrong shape {val.shape}'
        # SGD with lr=0.1 and grad=1 for 3 steps -> 1.0 - 0.1*3 = 0.7
        if name == 'sgd':
            expected = t.tensor([0.7, 0.7])
            assert t.allclose(val, expected, atol=1e-5), f'SGD: expected {expected}, got {val}'
        # Adam/AdamW should have moved away from init
        assert not t.allclose(val, init, atol=1e-5), f'{name} param should have moved'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

OPTIM_MAP = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def build_optimizer(name: str, params, lr: float) -> t.optim.Optimizer:
    return OPTIM_MAP[name](params, lr=lr)

def run_sweep(param_init: t.Tensor, lr: float, steps: int) -> dict:
    results = {}
    for name in OPTIM_MAP:
        p = param_init.clone().detach().requires_grad_(True)
        opt = build_optimizer(name, [p], lr=lr)
        for _ in range(steps):
            opt.zero_grad()
            p.grad = t.ones_like(p)
            opt.step()
        results[name] = p.detach().clone()
    return results

# --- run it ---
t.manual_seed(0)
init = t.tensor([1.0, 1.0])
results = run_sweep(init, lr=0.1, steps=3)
for name, val in results.items():
    print(f'{name}: {val}')
```
</details>